In [1]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# Practical 9: Bayesian Network for Heart Disease

import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination


# Step 1: Load the dataset


data = pd.read_csv("heart_disease.csv")

# Clean column names
data.columns = data.columns.str.strip()

print("Dataset loaded successfully.")
print("Available columns:", data.columns.tolist())


# Step 2: Select the required columns
data = data[["cp", "exang", "thalach", "diagnosis"]].copy()


# Step 3: Convert columns into numeric values
columns = ["cp", "exang", "thalach", "diagnosis"]

for column in columns:
    data[column] = pd.to_numeric(data[column], errors="coerce")

# Remove rows with missing required values
data = data.dropna().copy()


# Step 4: Convert diagnosis into two classes
data["diagnosis"] = (data["diagnosis"] > 0).astype(int)


# Step 5: Group maximum heart rate into categories
data["thalach_group"] = pd.cut(
    data["thalach"],
    bins=[0, 120, 160, float("inf")],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

# Remove rows without a heart rate category
data = data.dropna(subset=["thalach_group"]).copy()


# Convert categorical variables into strings
data["cp"] = data["cp"].astype(int).astype(str)
data["exang"] = data["exang"].astype(int).astype(str)
data["thalach_group"] = data["thalach_group"].astype(str)

# Keep only the variables required by the network
data = data[["cp", "exang", "thalach_group", "diagnosis"]].copy()

print("\nPrepared Dataset:")
print(data.head())


# Step 6: Create the Bayesian Network
model = DiscreteBayesianNetwork([
    ("cp", "diagnosis"),
    ("exang", "diagnosis"),
    ("thalach_group", "diagnosis")
])


# Step 7: Estimate the probability distributions
estimator = BayesianEstimator(model, data)

cpds = estimator.get_parameters(prior_type="BDeu")

# Add the probability distributions to the model
for cpd in cpds:
    model.add_cpds(cpd)

# Check whether the model is valid
if model.check_model():
    print("\nBayesian Network created successfully.")


# Step 8: Create the inference engine
inference = VariableElimination(model)


# Step 9: Enter sample patient information
evidence = {
    "cp": "4",
    "exang": "1",
    "thalach_group": "Low"
}


# Step 10: Calculate diagnosis probabilities
result = inference.query(
    variables=["diagnosis"],
    evidence=evidence,
    show_progress=False
)

print("\nPatient Information:")
print(evidence)

print("\nDiagnosis Probabilities:")
print(result)


# Step 11: Find the class with the highest probability
states = result.state_names["diagnosis"]

index = int(result.values.argmax())
predicted_class = states[index]

print("\nPredicted Class:")

if int(predicted_class) == 1:
    print("Disease present")
else:
    print("No disease")

C:\Users\akank\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset loaded successfully.
Available columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'diagnosis']

Prepared Dataset:
  cp exang thalach_group  diagnosis
0  1     0        Medium          0
1  4     1           Low          1
2  4     1        Medium          1
3  3     0          High          0
4  2     0          High          0

Bayesian Network created successfully.

Patient Information:
{'cp': '4', 'exang': '1', 'thalach_group': 'Low'}

Diagnosis Probabilities:
+--------------+------------------+
| diagnosis    |   phi(diagnosis) |
+==============+==================+
| diagnosis(0) |           0.0992 |
+--------------+------------------+
| diagnosis(1) |           0.9008 |
+--------------+------------------+

Predicted Class:
Disease present
